# RHI Live Runtime v14 — Robust Model-Call Recursive Agent

Δ **Purpose:** v13 isolated the real tear: the model loaded on CUDA, but generation failed before the controller could test actual branch outputs.

v13 correctly blocked false collapse:

$$
\text{fallback output} \neq \Psi
$$

v14 fixes the model call by avoiding the fragile positional `model.generate(input_ids, ...)` path. Some tokenizer/chat-template combinations return a `BatchEncoding`-like object instead of a plain tensor; passing that positionally causes the observed `AttributeError` at `inputs_tensor.shape`.

v14 uses the safer path:

$$
\text{messages} \rightarrow \text{chat template string} \rightarrow \text{tokenizer(...)} \rightarrow \text{model.generate(**inputs)}
$$

Key rule:

$$
\text{If model generation fails, state}=\Omega_{\text{model}},\quad \text{not fake }\Psi.
$$


In [1]:
# Optional install cell. Run only if a package is missing.
# %pip install -U pandas numpy torch transformers accelerate safetensors sentencepiece


In [2]:
from __future__ import annotations

import os
import re
import json
import math
import time
import uuid
import random
import traceback
from dataclasses import dataclass, asdict, field
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v14_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v14_" + uuid.uuid4().hex[:10]

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("RUN_ID:", RUN_ID)


ROOT: D:\Nexus\Nexus Mark 9\NoteBooks
OUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs
RUN_ID: rhi_v14_4e1d35182c


In [3]:
MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")

LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True

MAX_NEW_TOKENS = 420
TEMPERATURE = 0.35
TOP_P = 0.90
MAX_RECURSION_DEPTH = 2

PSI_MIN = 0.58
MARGIN_MIN = 0.045
PROMPT_FIT_MIN = 0.28
QUALITY_MIN = 0.42
BOILERPLATE_MAX = 0.30
TRACE_MIN = 0.45

print("MODEL_ID_OR_PATH:", MODEL_ID_OR_PATH)
print("LOAD_REAL_MODEL:", LOAD_REAL_MODEL)
print("REQUIRE_MODEL_FOR_PSI:", REQUIRE_MODEL_FOR_PSI)


MODEL_ID_OR_PATH: Qwen/Qwen2.5-1.5B-Instruct
LOAD_REAL_MODEL: True
REQUIRE_MODEL_FOR_PSI: True


In [4]:
# Model loading and robust generation.
# v14 fix: never pass a BatchEncoding or tokenizer output positionally into model.generate.
# Always use model.generate(**inputs).

tokenizer = None
model = None
MODEL_READY = False
MODEL_GENERATION_READY = False
MODEL_ERROR = None
DEVICE_INFO = {}


def infer_model_device():
    import torch
    if model is None:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def move_batch_to_device(batch, device):
    moved = {}
    for k, v in batch.items():
        if hasattr(v, "to"):
            moved[k] = v.to(device)
        else:
            moved[k] = v
    return moved


def try_load_model(model_id_or_path: str) -> bool:
    global tokenizer, model, MODEL_READY, MODEL_ERROR, DEVICE_INFO

    if not LOAD_REAL_MODEL:
        MODEL_ERROR = "LOAD_REAL_MODEL=False"
        print("Model loading disabled.")
        return False

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        DEVICE_INFO["torch_version"] = torch.__version__
        DEVICE_INFO["cuda_available"] = bool(torch.cuda.is_available())
        DEVICE_INFO["device_count"] = int(torch.cuda.device_count())
        if torch.cuda.is_available():
            DEVICE_INFO["gpu_name"] = torch.cuda.get_device_name(0)
            DEVICE_INFO["cuda_version"] = torch.version.cuda

        print("Torch/CUDA:", DEVICE_INFO)

        tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, trust_remote_code=True)
        if tokenizer.pad_token_id is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        kwargs = dict(
            trust_remote_code=True,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True,
        )
        try:
            model = AutoModelForCausalLM.from_pretrained(model_id_or_path, dtype=dtype, **kwargs)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(model_id_or_path, torch_dtype=dtype, **kwargs)

        if not torch.cuda.is_available():
            model.to(torch.device("cpu"))

        model.eval()
        MODEL_READY = True
        MODEL_ERROR = None
        print("MODEL_READY:", MODEL_READY)
        print("INFER_DEVICE:", infer_model_device())
        return True

    except Exception as e:
        MODEL_READY = False
        MODEL_ERROR = "".join(traceback.format_exception_only(type(e), e)).strip()
        print("MODEL LOAD FAILED.")
        print(MODEL_ERROR)
        return False


def render_messages(messages: List[Dict[str, str]]) -> str:
    # Render chat messages to a string robustly across tokenizer versions.
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            rendered = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            if isinstance(rendered, str) and rendered.strip():
                return rendered
        except Exception as e:
            print("apply_chat_template string render failed; using manual template:", type(e).__name__, e)

    parts = []
    for m in messages:
        role = str(m.get("role", "user")).upper()
        content = str(m.get("content", ""))
        parts.append(f"{role}:\n{content}")
    parts.append("ASSISTANT:\n")
    return "\n\n".join(parts)


def raw_model_generate(messages: List[Dict[str, str]], max_new_tokens: int = 80, sample: bool = False) -> str:
    import torch
    if not MODEL_READY:
        raise RuntimeError("Model is not loaded.")

    device = infer_model_device()
    rendered = render_messages(messages)
    inputs = tokenizer(rendered, return_tensors="pt")
    inputs = move_batch_to_device(inputs, device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if sample:
        gen_kwargs.update(dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P))
    else:
        gen_kwargs.update(dict(do_sample=False))

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    input_len = inputs["input_ids"].shape[-1]
    gen = out[0][input_len:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


_ = try_load_model(MODEL_ID_OR_PATH)

try:
    smoke = raw_model_generate([
        {"role": "system", "content": "You are a runtime smoke test."},
        {"role": "user", "content": "Reply with one short sentence containing the word READY."},
    ], max_new_tokens=40, sample=False)
    MODEL_GENERATION_READY = bool(smoke.strip())
    print("MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
    print("SMOKE:", smoke)
except Exception as e:
    MODEL_GENERATION_READY = False
    MODEL_ERROR = "".join(traceback.format_exception_only(type(e), e)).strip()
    print("MODEL GENERATION FAILED.")
    print(MODEL_ERROR)
    print(traceback.format_exc())


Torch/CUDA: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'gpu_name': 'NVIDIA GeForce RTX 4060', 'cuda_version': '12.6'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL_READY: True
INFER_DEVICE: cuda:0
MODEL_GENERATION_READY: True
SMOKE: I am ready to execute any given tasks or tests as requested.


In [5]:
STOPWORDS = {
    "the","a","an","and","or","but","if","then","else","of","to","in","on","for","with","by","as",
    "is","are","was","were","be","being","been","it","this","that","these","those","from","at",
    "into","out","about","so","because","therefore","than","not","no","yes","do","does","did",
    "can","could","should","would","will","just","they","them","their","you","your","we","our",
    "i","me","my","he","she","his","her","its","what","how","why","when"
}

NEXUS_SURFACE_TERMS = {
    "nexus","contract","carrier","domain","boundary","collapse","shape","value","slot","need",
    "forbidden","neighbor","operational","recursive","recursion","krrb","omega","psi","field",
    "fold","runtime","phase","lock","audit","trace","signal","evidence","branch","repair",
    "candidate","construct","verify","counter","prompt"
}

BOILERPLATE_PHRASES = [
    "the prompt is asking",
    "contract-first answer",
    "the correct flow is prompt",
    "answer should not be a noun lookup",
    "construct the inverse shape",
    "forms a need-slot before acting",
    "diagnostic fallback",
    "this is not a model answer",
]


def words(text: str, remove_nexus_surface: bool = False) -> List[str]:
    toks = re.findall(r"[a-zA-Z0-9_ΔΨΩ⊕↻⊥]+", str(text).lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    if remove_nexus_surface:
        toks = [t for t in toks if t not in NEXUS_SURFACE_TERMS]
    return toks


def wordset(text: str, remove_nexus_surface: bool = False) -> set:
    return set(words(text, remove_nexus_surface=remove_nexus_surface))


def jaccard_text(a: str, b: str, remove_nexus_surface: bool = False) -> float:
    wa, wb = wordset(a, remove_nexus_surface), wordset(b, remove_nexus_surface)
    if not wa and not wb:
        return 1.0
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / max(1, len(wa | wb))


def contains_any(text: str, terms: List[str]) -> bool:
    s = str(text).lower()
    return any(str(t).lower() in s for t in terms)


def clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, float(x)))


def harmonic_mean(vals: List[float], eps: float = 1e-9) -> float:
    vals = [max(eps, float(v)) for v in vals]
    return len(vals) / sum(1.0 / v for v in vals)


def field_hit_score(text: str, field_terms: List[str], max_terms: int = 10) -> float:
    if not field_terms:
        return 0.5
    s = str(text).lower()
    uniq = []
    for t in field_terms:
        t = str(t).lower().strip()
        if t and t not in uniq:
            uniq.append(t)
    uniq = uniq[:max_terms]
    hits = sum(1 for term in uniq if term in s)
    return clamp(hits / max(1, len(uniq)))


def boilerplate_penalty(text: str) -> float:
    s = str(text).lower()
    phrase_hits = sum(1 for p in BOILERPLATE_PHRASES if p in s)
    repeated_contract_words = sum(s.count(t) for t in ["contract", "collapse", "slot", "operational", "branch"])
    phrase_pen = phrase_hits / max(1, len(BOILERPLATE_PHRASES))
    repeat_pen = clamp(max(0, repeated_contract_words - 8) / 20)
    return clamp(0.70 * phrase_pen + 0.30 * repeat_pen)


In [6]:
SHAPE_TEMPLATES = {
    "CONTRACT": {"triggers": ["contract", "before", "intent", "tool", "agent", "plan", "spec", "interface"], "needs": ["intent", "boundary", "tool", "before", "select", "gate"]},
    "GROOVE": {"triggers": ["train", "lora", "qlora", "adapter", "fine tune", "weights", "groove", "model"], "needs": ["adapter", "low-rank", "weights", "delta", "dataset", "loss", "eval"]},
    "SEARCH": {"triggers": ["search", "retrieve", "retrieval", "find", "query", "lookup", "index", "rag"], "needs": ["query", "retrieve", "candidate", "rank", "verify", "evidence"]},
    "REPAIR": {"triggers": ["fix", "repair", "error", "failed", "broken", "bug", "traceback", "syntaxerror", "nameerror"], "needs": ["failure", "cause", "patch", "test", "rerun", "trace"]},
    "MEMORY": {"triggers": ["remember", "memory", "recall", "lost", "state", "context", "continuity"], "needs": ["state", "trace", "retrieve", "preserve", "update", "continuity"]},
    "BOUNDARY": {"triggers": ["boundary", "limit", "forbidden", "constraint", "safety", "gate", "reject"], "needs": ["boundary", "reject", "constraint", "preserve", "violate", "gate"]},
    "RECURSE": {"triggers": ["recursive", "recursion", "again", "loop", "fold", "iterate", "turn"], "needs": ["recursive", "branch", "feedback", "repair", "collapse", "omega"]},
    "TOOL": {"triggers": ["tool", "api", "call", "function", "agent", "execute", "act"], "needs": ["tool", "input", "output", "side-effect", "verify"]},
}


def detect_shape_template(prompt: str) -> List[str]:
    p = str(prompt).lower()
    active = []
    for name, cfg in SHAPE_TEMPLATES.items():
        if any(t in p for t in cfg["triggers"]):
            active.append(name)
    return active or ["GENERAL"]


def shape_mass(text: str, active_templates: List[str]) -> Dict[str, float]:
    masses = {}
    for name in active_templates:
        if name == "GENERAL":
            continue
        needs = SHAPE_TEMPLATES[name]["needs"]
        masses[name] = sum(1 for n in needs if n.lower() in str(text).lower()) / max(1, len(needs))
    if not masses:
        masses["GENERAL"] = 0.5
    return masses


def shape_score(text: str, active_templates: List[str]) -> float:
    masses = shape_mass(text, active_templates)
    return clamp(sum(masses.values()) / max(1, len(masses)))


@dataclass
class NeedSlotContract:
    prompt: str
    active_templates: List[str]
    inverse_need: str
    preserved_function: str
    boundary_conditions: List[str]
    domain_carrier: List[str]
    forbidden_neighbors: List[str]
    collapse_target: str
    repair_history: List[Dict[str, Any]] = field(default_factory=list)


def extract_domain_terms(prompt: str, max_terms: int = 14) -> List[str]:
    ws = words(prompt, remove_nexus_surface=True)
    counts = Counter(ws)
    return [w for w, _ in counts.most_common(max_terms)]


def infer_forbidden_neighbors(active: List[str]) -> List[str]:
    forb = set()
    if "TOOL" in active or "CONTRACT" in active:
        forb.update(["tool-first action", "premature execution", "api reflex", "surface task completion"])
    if "GROOVE" in active:
        forb.update(["full retrain reflex", "weight churn", "dataset worship", "loss-only tuning"])
    if "SEARCH" in active:
        forb.update(["noun lookup", "keyword matching", "unverified retrieval", "search without verifier"])
    if "REPAIR" in active:
        forb.update(["blanket rewrite", "threshold fiddling", "silent failure", "patch without test"])
    if "MEMORY" in active:
        forb.update(["stateless answer", "context amnesia", "surface recall", "summary as memory"])
    if "BOUNDARY" in active:
        forb.update(["unsafe override", "constraint erasure", "boundary confusion"])
    if "RECURSE" in active:
        forb.update(["linear pipeline", "single branch", "dead loop", "nested sweep masquerading as recursion"])
    if not forb:
        forb.update(["surface label", "generic explanation", "noun-only answer"])
    return sorted(forb)


def build_contract(prompt: str, repair_history: Optional[List[Dict[str, Any]]] = None) -> NeedSlotContract:
    active = detect_shape_template(prompt)
    domain_terms = extract_domain_terms(prompt)

    inverse_need = "construct the missing operational slot implied by the prompt; select or generate only answers that preserve the required operation"

    preserved = []
    if "TOOL" in active or "CONTRACT" in active:
        preserved.append("form contract before tool use")
    if "GROOVE" in active:
        preserved.append("shape model behavior through low-rank update without overwriting the base model")
    if "SEARCH" in active:
        preserved.append("retrieve by inverse operational fit when no noun match exists")
    if "REPAIR" in active:
        preserved.append("repair the failed dimension and rerun")
    if "MEMORY" in active:
        preserved.append("preserve trace continuity across turns rather than compressing state into summary text")
    if "RECURSE" in active:
        preserved.append("branch recursively until Ψ collapse or Ω residue")
    if not preserved:
        preserved.append("preserve the prompt's verb-level operation")

    boundaries = [
        "do not collapse on shared framework vocabulary alone",
        "require answer origin from the real model when model mode is enabled",
        "require prompt-grounded evidence for the selected answer",
        "prefer Ω over false Ψ when top branches disagree operationally",
        "preserve base answer when controller evidence is weak",
    ]

    return NeedSlotContract(
        prompt=prompt,
        active_templates=active,
        inverse_need=inverse_need,
        preserved_function="; ".join(preserved),
        boundary_conditions=boundaries,
        domain_carrier=domain_terms,
        forbidden_neighbors=infer_forbidden_neighbors(active),
        collapse_target="one executable answer with model origin, contract fit, prompt grounding, and trace sufficient to debug",
        repair_history=repair_history or [],
    )


def contract_to_text(c: NeedSlotContract) -> str:
    return (
        f"ACTIVE_TEMPLATES: {', '.join(c.active_templates)}\n"
        f"INVERSE_NEED: {c.inverse_need}\n"
        f"PRESERVED_FUNCTION: {c.preserved_function}\n"
        f"BOUNDARY_CONDITIONS: {' | '.join(c.boundary_conditions)}\n"
        f"DOMAIN_CARRIER: {', '.join(c.domain_carrier)}\n"
        f"FORBIDDEN_NEIGHBORS: {' | '.join(c.forbidden_neighbors)}\n"
        f"COLLAPSE_TARGET: {c.collapse_target}\n"
        f"REPAIR_HISTORY: {json.dumps(c.repair_history, ensure_ascii=False)}"
    )


def contract_field_terms(contract: NeedSlotContract) -> Dict[str, List[str]]:
    return {
        "need": words(contract.inverse_need, remove_nexus_surface=True),
        "function": words(contract.preserved_function, remove_nexus_surface=True),
        "boundary": words(" ".join(contract.boundary_conditions), remove_nexus_surface=True),
        "domain": contract.domain_carrier,
        "forbidden": words(" ".join(contract.forbidden_neighbors), remove_nexus_surface=True),
        "collapse": words(contract.collapse_target, remove_nexus_surface=True),
    }


In [7]:
BRANCH_SYSTEMS = {
    "construct": "You are the constructor branch. Build the answer from the missing operational slot first. Do not start with labels. Make the answer specific to the prompt.",
    "verify": "You are the verifier branch. Test the answer against need, function, boundary, trap, and collapse. Reject vocabulary agreement when operation differs.",
    "repair": "You are the repair branch. Identify the failed observable and patch only that dimension. Do not blanket-rewrite.",
    "counter": "You are the counter-branch. Name the strongest wrong path and explain why it fails. Then give the corrected path.",
}


def deterministic_branch(prompt: str, contract: NeedSlotContract, branch_name: str, reason: str = "fallback") -> Dict[str, Any]:
    if branch_name == "construct":
        text = f"Diagnostic fallback for construct. Prompt domain: {', '.join(contract.domain_carrier[:6])}. Preserved function: {contract.preserved_function}. This is not a model answer; it exists only to keep the controller inspectable."
    elif branch_name == "verify":
        text = "Diagnostic fallback for verify. Check whether the candidate preserves the function, rejects forbidden neighbors, and stays grounded in the prompt rather than shared vocabulary."
    elif branch_name == "repair":
        text = "Diagnostic fallback for repair. If Ψ is blocked, extract Ω as the weakest observable and regenerate only that dimension."
    elif branch_name == "counter":
        text = "Diagnostic fallback for counter. The wrong path is collapsing on boilerplate or model failure while pretending success."
    else:
        text = "Diagnostic fallback."
    return {"branch": branch_name, "answer": text, "origin": reason, "generation_error": MODEL_ERROR}


def model_generate_one(prompt: str, contract: NeedSlotContract, branch_name: str) -> Dict[str, Any]:
    if not MODEL_GENERATION_READY:
        return deterministic_branch(prompt, contract, branch_name, reason="fallback_model_not_ready")

    user = (
        "PROMPT:\n" + prompt.strip() + "\n\n"
        "NEED-SLOT CONTRACT:\n" + contract_to_text(contract) + "\n\n"
        "Rules:\n"
        "1. Do not say 'the prompt is asking'.\n"
        "2. Do not recite the whole pipeline unless the prompt asks for a pipeline.\n"
        "3. Use at least two prompt-domain terms, not just framework terms.\n"
        "4. Give one compact operational answer.\n"
        "5. Make the answer testable.\n"
    )
    messages = [{"role": "system", "content": BRANCH_SYSTEMS[branch_name]}, {"role": "user", "content": user}]
    try:
        text = raw_model_generate(messages, max_new_tokens=MAX_NEW_TOKENS, sample=True)
        if not text.strip():
            return deterministic_branch(prompt, contract, branch_name, reason="fallback_empty_generation")
        return {"branch": branch_name, "answer": text.strip(), "origin": "model", "generation_error": None}
    except Exception as e:
        err = "".join(traceback.format_exception_only(type(e), e)).strip()
        return {**deterministic_branch(prompt, contract, branch_name, reason="fallback_error"), "generation_error": err}


def generate_candidates(prompt: str, contract: NeedSlotContract) -> List[Dict[str, Any]]:
    return [model_generate_one(prompt, contract, b) for b in BRANCH_SYSTEMS]


In [8]:
def answer_operational_audit(prompt: str, contract: NeedSlotContract, answer: str, origin: str) -> Dict[str, Any]:
    active = contract.active_templates
    fields = contract_field_terms(contract)
    a = str(answer).lower()
    F_need = clamp(0.55 * field_hit_score(answer, fields["need"]) + 0.45 * field_hit_score(answer, fields["domain"]))
    F_function = clamp(0.65 * field_hit_score(answer, fields["function"]) + 0.35 * sum([contains_any(a, ["preserve", "maintain", "function", "operation", "before", "after"]), contains_any(a, ["execute", "candidate", "select", "verify", "state", "update"])]) / 2)
    F_boundary = clamp(0.60 * field_hit_score(answer, fields["boundary"]) + 0.40 * sum([contains_any(a, ["boundary", "constraint", "gate", "reject", "protect", "forbidden"]), contains_any(a, ["false", "wrong", "weak", "premature", "surface", "boilerplate"])]) / 2)
    forbidden_hit = field_hit_score(answer, fields["forbidden"])
    trap_language = sum([contains_any(a, ["not", "instead", "wrong", "fails", "reject", "avoid", "forbidden"]), contains_any(a, ["tool-first", "surface", "generic", "threshold", "noun", "keyword", "boilerplate"])]) / 2
    F_trap = clamp(0.45 * forbidden_hit + 0.55 * trap_language)
    F_collapse = clamp(0.50 * field_hit_score(answer, fields["collapse"]) + 0.50 * sum([contains_any(a, ["because", "therefore", "so", "result", "answer"]), contains_any(a, ["one", "single", "executable", "run", "test", "trace"])]) / 2)
    F_shape = shape_score(answer, active)
    F_prompt = field_hit_score(answer, words(prompt, remove_nexus_surface=True), max_terms=12)
    B_penalty = boilerplate_penalty(answer)
    O_model = 1.0 if origin == "model" else 0.0
    hot = clamp((F_need + F_function + F_shape + F_prompt) / 4)
    cold = clamp((F_boundary + F_trap + F_collapse + (1.0 - B_penalty)) / 4)
    hotcold_balance = clamp(1.0 - abs(hot - cold))
    quality_hmean = harmonic_mean([F_need, F_function, F_boundary, F_trap, F_collapse, max(0.001, F_prompt)])
    quality_mean = float(np.mean([F_need, F_function, F_boundary, F_trap, F_collapse, F_prompt]))
    return {"F_need": F_need, "F_function": F_function, "F_boundary": F_boundary, "F_trap": F_trap, "F_collapse": F_collapse, "F_shape": F_shape, "F_prompt": F_prompt, "boilerplate_penalty": B_penalty, "O_model": O_model, "hot": hot, "cold": cold, "hotcold_balance": hotcold_balance, "quality_hmean": quality_hmean, "quality_mean": quality_mean, "active_templates": active, "shape_mass": shape_mass(answer, active)}


def trace_sufficiency(answer: str, audit: Dict[str, Any], contract: NeedSlotContract, origin: str) -> float:
    a = str(answer).lower()
    bits = [origin == "model", audit["F_prompt"] >= PROMPT_FIT_MIN, audit["quality_hmean"] >= QUALITY_MIN, audit["boilerplate_penalty"] <= BOILERPLATE_MAX, contains_any(a, ["because", "therefore", "so", "prevents", "requires", "fails", "works"]), contains_any(a, contract.domain_carrier[:8])]
    return clamp(sum(1 for b in bits if b) / len(bits))


def branch_score(prompt: str, contract: NeedSlotContract, branch: Dict[str, Any]) -> Dict[str, Any]:
    answer = branch["answer"]
    origin = branch.get("origin", "unknown")
    audit = answer_operational_audit(prompt, contract, answer, origin)
    fields = contract_field_terms(contract)
    contract_stance = float(np.mean([field_hit_score(answer, fields["need"]), field_hit_score(answer, fields["function"]), field_hit_score(answer, fields["boundary"]), field_hit_score(answer, fields["domain"]), field_hit_score(answer, fields["collapse"])]))
    trace = trace_sufficiency(answer, audit, contract, origin)
    score = clamp(0.27 * audit["quality_hmean"] + 0.18 * contract_stance + 0.16 * audit["F_shape"] + 0.15 * trace + 0.10 * audit["hotcold_balance"] + 0.10 * audit["F_prompt"] + 0.10 * audit["O_model"] - 0.06 * audit["boilerplate_penalty"])
    return {**branch, "score": score, "contract_stance": contract_stance, "trace_sufficiency": trace, "audit": audit}


def score_candidates(prompt: str, contract: NeedSlotContract, candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = [branch_score(prompt, contract, c) for c in candidates]
    flat = []
    for r in rows:
        a = r["audit"]
        flat.append({"branch": r["branch"], "origin": r.get("origin"), "score": r["score"], "contract_stance": r["contract_stance"], "trace_sufficiency": r["trace_sufficiency"], "quality_hmean": a["quality_hmean"], "F_need": a["F_need"], "F_function": a["F_function"], "F_boundary": a["F_boundary"], "F_trap": a["F_trap"], "F_collapse": a["F_collapse"], "F_shape": a["F_shape"], "F_prompt": a["F_prompt"], "boilerplate_penalty": a["boilerplate_penalty"], "O_model": a["O_model"], "hot": a["hot"], "cold": a["cold"], "hotcold_balance": a["hotcold_balance"], "generation_error": r.get("generation_error"), "answer": r["answer"]})
    return pd.DataFrame(flat).sort_values("score", ascending=False).reset_index(drop=True)


In [9]:
def direct_gate(df: pd.DataFrame) -> Dict[str, Any]:
    top = df.iloc[0]
    second_score = float(df.iloc[1]["score"]) if len(df) > 1 else 0.0
    margin = float(top["score"] - second_score)
    failed = []
    if float(top["score"]) < PSI_MIN: failed.append("score")
    if margin < MARGIN_MIN: failed.append("margin")
    if float(top["trace_sufficiency"]) < TRACE_MIN: failed.append("trace")
    if float(top["quality_hmean"]) < QUALITY_MIN: failed.append("quality")
    if float(top["F_prompt"]) < PROMPT_FIT_MIN: failed.append("prompt_grounding")
    if float(top["boilerplate_penalty"]) > BOILERPLATE_MAX: failed.append("boilerplate")
    if REQUIRE_MODEL_FOR_PSI and top["origin"] != "model": failed.append("model_origin")
    return {"ok": len(failed) == 0, "reason": "direct_model_grounded_collapse" if len(failed) == 0 else "no_direct_collapse", "failed": failed, "margin": margin, "top_score": float(top["score"]), "top_origin": str(top["origin"]), "trace_sufficiency": float(top["trace_sufficiency"]), "quality_hmean": float(top["quality_hmean"]), "F_prompt": float(top["F_prompt"]), "boilerplate_penalty": float(top["boilerplate_penalty"])}


def weakest_observable(row: pd.Series) -> str:
    vals = {"need": row["F_need"], "function": row["F_function"], "boundary": row["F_boundary"], "trap": row["F_trap"], "collapse": row["F_collapse"], "prompt_grounding": row["F_prompt"], "model_origin": row["O_model"]}
    return min(vals.items(), key=lambda kv: kv[1])[0]


def repair_contract(contract: NeedSlotContract, failed: List[str], top_row: pd.Series) -> NeedSlotContract:
    weakest = weakest_observable(top_row)
    hist = list(contract.repair_history)
    hist.append({"failed_gate": failed, "weakest_observable": weakest, "winner_branch": str(top_row["branch"]), "winner_origin": str(top_row["origin"]), "winner_score": float(top_row["score"])})
    prompt = contract.prompt + f"\n\nREPAIR TARGET: strengthen {weakest}; do not change unrelated dimensions."
    return build_contract(prompt, repair_history=hist)


def run_rhi_v14(prompt: str, depth: int = 0, repair_history: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    contract = build_contract(prompt, repair_history=repair_history)
    candidates = generate_candidates(prompt, contract)
    df = score_candidates(prompt, contract, candidates)
    gate = direct_gate(df)
    trace_entry = {"depth": depth, "contract": asdict(contract), "scores": df.to_dict(orient="records"), "direct_gate": gate}
    top = df.iloc[0]
    common = {"run_id": RUN_ID, "prompt": prompt, "depth": depth, "winner_branch": str(top["branch"]), "winner_origin": str(top["origin"]), "winner_score": float(top["score"]), "answer": str(top["answer"]), "contract": asdict(contract), "trace": [trace_entry], "model_generation_ready": MODEL_GENERATION_READY, "model_error": MODEL_ERROR}
    if gate["ok"]:
        result = {**common, "state": "Ψ", "reason": gate["reason"]}
    else:
        if "model_origin" in gate["failed"] and not MODEL_GENERATION_READY:
            result = {**common, "state": "Ω", "reason": "model_generation_failed"}
        elif depth >= MAX_RECURSION_DEPTH:
            result = {**common, "state": "Ω", "reason": "max_depth_residue"}
        else:
            repaired = repair_contract(contract, gate["failed"], top)
            sub = run_rhi_v14(repaired.prompt, depth=depth + 1, repair_history=repaired.repair_history)
            sub["trace"] = [trace_entry] + sub.get("trace", [])
            result = sub
    stem = f"{RUN_ID}_{abs(hash(prompt)) % (10**10):010d}"
    out_json = OUT_DIR / f"{stem}_result.json"
    out_scores = OUT_DIR / f"{stem}_scores.csv"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    df.to_csv(out_scores, index=False)
    print("saved:", out_json)
    print("saved:", out_scores)
    return result


In [10]:
TEST_PROMPTS = [
    "explain why current AI agents fail when they use tools before forming a contract",
    "fix a recursive AI controller that collapses because two branches share vocabulary but disagree operationally",
    "how should a LoRA adapter train a slot-builder without overwriting the base model",
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
    "explain memory in an agent as trace continuity rather than a text summary",
]

results = []
for p in TEST_PROMPTS:
    print("\n" + "="*90)
    print("PROMPT:", p)
    r = run_rhi_v14(p)
    results.append(r)
    print("STATE:", r["state"], "REASON:", r["reason"], "ORIGIN:", r.get("winner_origin"), "DEPTH:", r.get("depth"))
    print("ANSWER:\n", r["answer"][:900])

summary = pd.DataFrame([{"prompt": r["prompt"], "state": r["state"], "reason": r["reason"], "depth": r["depth"], "winner_branch": r["winner_branch"], "winner_origin": r["winner_origin"], "winner_score": r["winner_score"], "model_generation_ready": r.get("model_generation_ready"), "model_error": r.get("model_error")} for r in results])
summary_path = OUT_DIR / f"{RUN_ID}_summary.csv"
summary.to_csv(summary_path, index=False)
print("\nSUMMARY SAVED:", summary_path)
display(summary)



PROMPT: explain why current AI agents fail when they use tools before forming a contract
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_8513990024_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_8513990024_scores.csv
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_9147625626_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_9147625626_scores.csv
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_9706691430_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_9706691430_scores.csv
STATE: Ω REASON: max_depth_residue ORIGIN: model DEPTH: 2
ANSWER:
 The current AI agents fail when using tools before forming a contract because their decision-making process lacks proper boundaries and constraints. The primary issue lies in the fact that these systems often prioritize tool usage over the formation of a binding a

,prompt,state,reason,depth,winner_branch,winner_origin,winner_score,model_generation_ready,model_error
0,explain why current AI agents fail when they u...,Ω,max_depth_residue,2,construct,model,0.520131,True,None
1,fix a recursive AI controller that collapses b...,Ψ,direct_model_grounded_collapse,2,verify,model,0.708188,True,None
2,how should a LoRA adapter train a slot-builder...,Ψ,direct_model_grounded_collapse,0,construct,model,0.748045,True,None
3,design a shape-first retrieval step where no n...,Ψ,direct_model_grounded_collapse,1,repair,model,0.785168,True,None
4,explain memory in an agent as trace continuity...,Ω,max_depth_residue,2,verify,model,0.413897,True,None


In [11]:
LIVE_PROMPT = "explain memory in an agent as trace continuity rather than a text summary"
live_result = run_rhi_v14(LIVE_PROMPT)
print(json.dumps({"state": live_result["state"], "reason": live_result["reason"], "winner_origin": live_result["winner_origin"], "depth": live_result["depth"], "answer": live_result["answer"][:1200]}, indent=2, ensure_ascii=False))


saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_6084593914_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_6084593914_scores.csv
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_5813548520_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_5813548520_scores.csv
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_6218969449_result.json
saved: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v14_outputs\rhi_v14_4e1d35182c_6218969449_scores.csv
{
  "state": "Ω",
  "reason": "max_depth_residue",
  "winner_origin": "model",
  "depth": 2,
  "answer": "To explain memory in an agent as trace continuity rather than a text summary:\n\nThe concept of memory in agents refers to their ability to retain and utilize past experiences and interactions to inform future actions. In this explanation, we emphasize that memory is not merely about storing information (as it mig